In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%PandasProfile
### cell 10 ###

gender_map = {
    'Male': 'Man',
    'Female': 'Woman',
    'A different identity': 'Prefer to self-describe',
    'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe',
    'Nonbinary': 'Prefer to self-describe',
    'Prefer not to say': 'Prefer to self-describe'
}


def combine_subset_of_data_from_multiple_years_22(question_of_interest, x_axis_title, include_2017=None):
    # List of (year, dataframe, column-to-use)
    year_dfs = [
        ('2022', responses_df_2022_cell10, question_of_interest),
        ('2021', responses_df_2021, question_of_interest),
        ('2020', responses_df_2020, question_of_interest),
        ('2019', responses_df_2019_cell10, question_of_interest),
        ('2018', responses_df_2018_cell10, question_of_interest),
    ]
    if include_2017 is not None:
        # 2017 uses a differently named column
        year_dfs.insert(0, ('2017', responses_df_2017, 'GenderSelect'))

    frames = []
    for year, df, col in year_dfs:
        # Replace categories, compute normalized percentages, round and sort
        pct = (
            df[col]
              .replace(gender_map)
              .value_counts(dropna=False, normalize=True)
              .mul(100)
              .round(1)
        )
        tmp = pct.rename_axis(col).reset_index(name='percentage')
        tmp['year'] = year
        frames.append(tmp)

    # Concatenate all years and rename the category column to x_axis_title
    df_combined = pd.concat(frames, ignore_index=True)
    df_combined = df_combined.rename(columns={
        question_of_interest: x_axis_title,
        'GenderSelect': x_axis_title
    })
    # Ensure final column order
    return df_combined[['percentage', 'year', x_axis_title]]

# Build and sort the combined DataFrame
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22,
    title_for_x_axis_cell22,
    include_2017='yes'
)
age_df_combined_cell22 = age_df_combined_cell22.sort_values(
    ['year', 'percentage'],
    ascending=True
)
age_df_combined_cell22.info()